###1. Загрузим легкую модель Qwen и на простом промте измерим качество.         
Полученный результат будем считать за baseline, который будем пытаться перебить с помощью   
  a) Улучшения промта (chain of thoughts, детализация и тд).  
  b) Более мощной модели по числу параметров.  
  с) Другой модели.  
  d) Перехода в fine-tuning.  


###2. Специфика данных:   
Эссе оценивается по 4-м критериям, итоговая оценка это просто среднее из них. Поэтому мы хотим от модели попадания в каждый из критериев, помимо общей оценки.

###3. Выбранные метрики качества:  
**MAE** - покажет в баллах на сколько ошибаемся в предсказании, что очень наглядно. Будем в идеале стремиться к тому, чтобы модель ошибалась менее, чем на 0.5 балла  
**RMSE** - учитываем выбросы. Считаем, что дать сильно неправильную оценку за эссе это критически плохо   
**MBE** - модель в реднем завышает или занижает предсказания? Будем стремиться к тому, чтобы модель минимально завышала/занижала оценки.

###4. Формирование пайплайна
Этот же код будем использовать при тесте других моделей/ промтов

In [ ]:
# !pip install -q transformers accelerate bitsandbytes pandas datasets torch -- расскомментировать если такой штуки еще не установлено

import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
print(f"Устройство: {torch.cuda.get_device_name(0)}")
print(f"Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU доступен: True
Устройство: Tesla T4
Память: 15.6 GB


In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

# Это наш выбранный датасет с каггла, в нем данные сразу разбиты на тест и трейн

!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d xuanhuynh233/ielts-dataset
!unzip ielts-dataset.zip

csv_path = 'train_final.csv'
df = pd.read_csv(csv_path)

print(df.head())
print(f"Размер датасета: {len(df)} строк.")

# Удалим строки, где 'essay' или 'band' отсутствуют,
# или 'band' не является числом.

initial_len = len(df)
df = df.dropna(subset=['essay', 'band'])
df = df[pd.to_numeric(df['band'], errors='coerce').notna()]
df['band'] = df['band'].astype(float) # Убедимся, что band это float
print(f"\nУдалено {initial_len - len(df)} строк с отсутствующими или некорректными 'essay'/'band'.")
print(f"Осталось {len(df)} строк после очистки.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: cannot stat '/content/drive/MyDrive/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/xuanhuynh233/ielts-dataset
License(s): MIT
ielts-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  ielts-dataset.zip
replace test_final.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace train_final.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
                                      band  \
0  7.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
1  5.0\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
2  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
3  5.5\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   
4    4\n\n\n\n\n\r\r\r\r\r\r\r\r\r\r\r\r\r   

                                              prompt  \
0  Interviews form the basic crite

In [ ]:
df

,band,prompt,essay,TR_Band,TR_Comment,CC_Band,CC_Comment,LR_Band,LR_Mistakes,LR_Corrections,LR_Comment,GRA_Band,GRA_Mistakes,GRA_Corrections,GRA_Comment,Overall_Band,General_Feedback,error
0,7.5,Interviews form the basic criteria for most la...,It is believed by some experts that the tradit...,8.0,The essay sufficiently addresses all parts of ...,8.0,The essay is logically structured and the info...,7.0,exams writing; his teamwork skill is measured;...,written exams; their teamwork skills are measu...,The essay demonstrates a sufficient range of v...,7.0,...his ability to do the work and their proble...,...their ability to do the work and their prob...,"A variety of complex structures is used, and t...",7.5,This is a strong response with a very clear st...,NaN
1,5.0,Interviews form the basic selecting criteria f...,Nowadays numerous huge firms allocate an inter...,5.0,The essay addresses the task only partially. I...,5.0,"The essay is presented with some organization,...",5.0,choosing criteria; up-to-the-minute; reap the ...,selection criterion/criteria; effective/reliab...,The writer attempts to use some less common vo...,5.0,having excellent interpersonal skills are; mak...,having excellent interpersonal skills is; effe...,The essay shows a limited range of sentence st...,5.0,The essay presents a clear position and follow...,NaN
2,5.5,Interview form the basic selection criteria fo...,The interview section is the most vital part o...,6.0,"You have addressed all parts of the question, ...",6.0,The essay has a clear structure with an introd...,5.0,Interview section; prospect for the specified ...,The interview stage; prospective candidate for...,You have used an adequate range of vocabulary ...,5.0,Interview form the basic selection criteria; w...,Interviews form the basic selection criteria; ...,You have attempted to use a mix of simple and ...,5.5,Your essay presents a clear opinion and is log...,NaN
3,5.5,Interviews form the basic selection criteria f...,It is argued that the best method to recruit e...,6.0,The essay addresses the prompt by discussing b...,5.5,"The essay is organized into paragraphs, and th...",5.0,acquired by the recruiter; choosing the best e...,possessed by the applicant; choosing the best ...,The range of vocabulary is limited but minimal...,5.0,who do not lacks in any single question; accor...,who do not lack an answer to any single questi...,The writer attempts to use a mix of simple and...,5.5,Your essay has a clear structure and you prese...,NaN
4,4.0,Interviews from the basic selecting criteria f...,Nowadays many companies conduct interviews bef...,4.0,The response addresses the task only in a mini...,4.0,"The essay presents information and ideas, but ...",4.0,chosing; intetviuvs; campaigns; sending them; ...,choosing; interviews; companies; hiring them; ...,The writer uses only basic and repetitive voca...,4.0,Interviews from the basic selecting criteria; ...,Interviews form the basic selection criteria; ...,Only a very limited range of sentence structur...,4.0,"Your essay attempts to address the prompt, but...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9828,8.0,Some universities offer online courses as an a...,In today's most developing and rapid-growing w...,8.0,The response fully and appropriately addresses...,8.0,The essay demonstrates skillful organization o...,8.0,some studies filed needs a certain level of la...,some fields of study need a certain level of l...,The writer utilizes a wide and flexible range ...,8.0,another side some thinks student have to atten...,"on the other hand, some think students have to...",A wide range of grammatical structures is used...,8.0,This is a well-developed and persuasive essay ...,NaN
9829,4.5,Some people think that the main purpose of sch...,Some people think that the main purpose of sch...,4.0,"The essay attempts to address the prompt, but ...",5.0,There is a basic paragraph structure with an i...,4.0,make off spring

## загрузка QWEN

In [ ]:
# !pip uninstall -y bitsandbytes
# !pip install bitsandbytes>=0.46.1

# Перезапустите сеанс

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Конфигурация 4-bit квантизации - чтобы быстро скачать и чтобы быстро работало (для пробы пера)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Легкая qwen с квантизацией скачивалась 2 минуты - норм

In [ ]:
# сразу все импорты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr
import re
from tqdm import tqdm
import torch

In [ ]:
# базовый промт
def create_ielts_prompt(essay_to_grade, example_essays):

    prompt = """You are an expert IELTS Writing Task 2 examiner. Grade essays using IELTS band descriptors (0-9, step 0.5).

IELTS CRITERIA (each 0-9):
1. TR (Task Response): How well the essay addresses the task, presents ideas, and supports arguments
2. CC (Coherence & Cohesion): Organization, paragraph structure, and linking devices
3. LR (Lexical Resource): Vocabulary range, accuracy, and appropriateness
4. GRA (Grammatical Range & Accuracy): Grammar variety, accuracy, and complexity

IMPORTANT:
- Use only 0.5 increments (e.g., 6.0, 6.5, 7.0)
- Overall Band = Average of all 4 criteria
- Provide realistic scores based on IELTS standards

Here are graded examples:

"""

    for i, example in enumerate(example_essays, 1):
        prompt += f"""Example {i}:
Essay: "{example['essay']}..."
TR_Band: {example['TR_Band']}
CC_Band: {example['CC_Band']}
LR_Band: {example['LR_Band']}
GRA_Band: {example['GRA_Band']}
Overall_Band: {example['Overall_Band']}
---

"""

    prompt += f"""Now, grade this essay following IELTS standards:

Essay: "{essay_to_grade}"

Provide your evaluation in this EXACT format:
TR_Band: X.X
CC_Band: X.X
LR_Band: X.X
GRA_Band: X.X
Overall_Band: X.X
Explanation: [Brief justification for each criterion]
"""

    return prompt

In [ ]:
# получим все оценки из ответа модели, чтобы считать по ним метрики качества

def extract_ielts_scores(response):

    # Извлечем 5 оценок {'TR': float, 'CC': float, 'LR': float, 'GRA': float, 'Overall': float}

    scores = {}

    patterns = {
        'TR': r'TR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'CC': r'CC[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'LR': r'LR[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'GRA': r'GRA[_\s]*Band:\s*(\d+(?:\.\d)?)',
        'Overall': r'Overall[_\s]*Band:\s*(\d+(?:\.\d)?)'
    }

    for criterion, pattern in patterns.items():
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            score = float(match.group(1))
            score = round(score * 2) / 2
            scores[criterion] = min(max(score, 0), 9)
        else:
            scores[criterion] = None

    if scores['Overall'] is None and all(scores[k] is not None for k in ['TR', 'CC', 'LR', 'GRA']):
        avg = np.mean([scores['TR'], scores['CC'], scores['LR'], scores['GRA']])
        scores['Overall'] = round(avg * 2) / 2

    return scores

In [ ]:
# функция оценки эссе по нашему промту
def grade_ielts_essay(essay, example_essays, model, tokenizer, max_tokens=600):


    prompt = create_ielts_prompt(essay, example_essays)
    # задаем роль
    messages = [
        {"role": "system", "content": "You are an expert IELTS Writing examiner with years of experience."},
        {"role": "user", "content": prompt}
    ]


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.2,  # Очень низкая для стабильности, была 0.3, выдавала разницу в качестве в 1 единцу MAE
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    return response

In [ ]:
# sample_size = 100  # отрабатывает час - больше не хочется повторять этот экспириенс
sample_size = 50
test_df = df.sample(sample_size, random_state=42).copy()

# будем давать 5 примеров оценненых эссе внутри промта
examples = df[~df.index.isin(test_df.index)].sample(5).to_dict('records')

predictions = {
    'TR': [],
    'CC': [],
    'LR': [],
    'GRA': [],
    'Overall': []
}
explanations = []

print(f"Оценка {sample_size} IELTS эссе...")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        # Генерация оценки моделью по промту
        response = grade_ielts_essay(
            row['essay'],
            examples,
            model,
            tokenizer,
            max_tokens=600
        )

        # Извлечение оценок из ответа модели
        scores = extract_ielts_scores(response)

        for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
            predictions[criterion].append(scores[criterion])

        explanations.append(response)

    except Exception as e:
        print(f"\n Ошибка на эссе {idx}: {e}")
        for criterion in predictions.keys():
            predictions[criterion].append(None)
        explanations.append("")

# Добавляем результаты в датафрейм
for criterion in ['TR', 'CC', 'LR', 'GRA', 'Overall']:
    test_df[f'pred_{criterion}'] = predictions[criterion]

test_df['explanation'] = explanations

print("\n Оценка завершена!")

Оценка 50 IELTS эссе...


100%|██████████| 50/50 [41:09<00:00, 49.39s/it]


 Оценка завершена!


#### На каждое эссе тратится около минуты, в будущем хотелось бы ускорить время работы
Тотал - 41 минута на 50 эссе

In [ ]:
# оцениваем качество модели

def calculate_ielts_metrics(actual, predicted):

    mask = ~(pd.isna(actual) | pd.isna(predicted))
    actual = np.array(actual)[mask]
    predicted = np.array(predicted)[mask]

    if len(actual) == 0:
        return None

    # MAE
    mae = mean_absolute_error(actual, predicted)

    # RMSE
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    # MBE чтобы понимать занижение/завышение оценок
    mbe = np.mean(predicted - actual)

    # Корреляция Спирмена
    correlation, _ = spearmanr(actual, predicted)

    # Точность
    acc_05 = np.mean(np.abs(predicted - actual) <= 0.5)
    acc_10 = np.mean(np.abs(predicted - actual) <= 1.0)

    # Процент точных совпадений (особенно важно для 0.5 шага)
    exact_match = np.mean(predicted == actual)

    # QWK
    from sklearn.metrics import cohen_kappa_score
    qwk = cohen_kappa_score(
        (actual * 2).astype(int),  # Переводим в целые (умножаем на 2)
        (predicted * 2).astype(int),
        weights='quadratic'
    )

    return {
        'MAE': mae,
        'RMSE': rmse,
        'MBE': mbe,
        'Correlation': correlation,
        'QWK': qwk,
        'Accuracy_±0.5': acc_05,
        'Accuracy_±1.0': acc_10,
        'Exact_Match': exact_match,
        'n_samples': len(actual)
    }


print("метрики качества")

criteria_mapping = {
    'TR': 'TR_Band',
    'CC': 'CC_Band',
    'LR': 'LR_Band',
    'GRA': 'GRA_Band',
    'Overall': 'Overall_Band'
}

all_metrics = {}

for criterion, actual_col in criteria_mapping.items():

    print(f" {criterion} ({actual_col})")
    actual = test_df[actual_col]
    predicted = test_df[f'pred_{criterion}']

    metrics = calculate_ielts_metrics(actual, predicted)

    if metrics:
        all_metrics[criterion] = metrics

        print(f"MAE:              {metrics['MAE']:.3f} bands")
        print(f"RMSE:             {metrics['RMSE']:.3f} bands")
        print(f"MBE:              {metrics['MBE']:+.3f} bands {'(завышение)' if metrics['MBE'] > 0 else '(занижение)' if metrics['MBE'] < 0 else '(нет смещения)'}")
        print(f"Корреляция:       {metrics['Correlation']:.3f}")
        print(f"QWK:              {metrics['QWK']:.3f}")
        print(f"Точность ±0.5:    {metrics['Accuracy_±0.5']:.1%}")
        print(f"Точность ±1.0:    {metrics['Accuracy_±1.0']:.1%}")
        print(f"Точное совпадение: {metrics['Exact_Match']:.1%}")
        print(f"Обработано:       {metrics['n_samples']} эссе")
    else:
        print(" Недостаточно данных для расчета метрик")


метрики качества
 TR (TR_Band)
MAE:              0.990 bands
RMSE:             1.151 bands
MBE:              -0.050 bands (занижение)
Корреляция:       0.227
QWK:              0.028
Точность ±0.5:    28.0%
Точность ±1.0:    74.0%
Точное совпадение: 14.0%
Обработано:       50 эссе
 CC (CC_Band)
MAE:              1.110 bands
RMSE:             1.317 bands
MBE:              -0.270 bands (занижение)
Корреляция:       0.187
QWK:              0.022
Точность ±0.5:    22.0%
Точность ±1.0:    68.0%
Точное совпадение: 18.0%
Обработано:       50 эссе
 LR (LR_Band)
MAE:              1.070 bands
RMSE:             1.279 bands
MBE:              +0.150 bands (завышение)
Корреляция:       0.136
QWK:              0.015
Точность ±0.5:    28.0%
Точность ±1.0:    68.0%
Точное совпадение: 18.0%
Обработано:       50 эссе
 GRA (GRA_Band)
MAE:              1.030 bands
RMSE:             1.243 bands
MBE:              +0.150 bands (завышение)
Корреляция:       0.146
QWK:              0.015
Точность ±0.5:    32.0%


In [ ]:
metrics_df = pd.DataFrame(all_metrics).T
metrics_df = metrics_df.round(3)
print(metrics_df[['MAE', 'RMSE', 'MBE', 'Correlation', 'QWK']])

          MAE   RMSE   MBE  Correlation    QWK
TR       0.99  1.151 -0.05        0.227  0.028
CC       1.11  1.317 -0.27        0.187  0.022
LR       1.07  1.279  0.15        0.136  0.015
GRA      1.03  1.243  0.15        0.146  0.015
Overall  1.07  1.219 -0.07        0.165  0.019


## Промежуточные выводы:

- Получили неплохое качество по MAE - по промежуточным оценкам и итоговым отклонение не больше 1,1 балла из 9
- Модель переоценивает максимум на 0,15 балла, в среднем недооценивает на 0,27 балла
- На метрике согласия QWK получили какой-то бред - модель работает практичсеки случайно 🤟

## Конец пайплайна
1) подача промта
2) работа модели на основе этого промта на случайной подвыборке из тестового датасета
3) оценка качества модели



Дальше подробнее посмотрим на вывод модели

In [ ]:
# выведем граничные метрики качества по всем оценкам
for criterion in criteria_names:
    actual = test_df[criteria_mapping[criterion]]
    predicted = test_df[f'pred_{criterion}']

    mask = actual.notna() & predicted.notna()
    errors = (predicted[mask] - actual[mask]).abs()

    print(f"\n{criterion}:")
    print(f"  Макс. ошибка: {errors.max():.1f} bands")
    print(f"  Мин. ошибка:  {errors.min():.1f} bands")
    print(f"  Медиана:      {errors.median():.2f} bands")
    print(f"  95 перцентиль: {errors.quantile(0.95):.2f} bands")


TR:
  Макс. ошибка: 2.0 bands
  Мин. ошибка:  0.0 bands
  Медиана:      1.00 bands
  95 перцентиль: 2.00 bands

CC:
  Макс. ошибка: 2.5 bands
  Мин. ошибка:  0.0 bands
  Медиана:      1.00 bands
  95 перцентиль: 2.00 bands

LR:
  Макс. ошибка: 2.0 bands
  Мин. ошибка:  0.0 bands
  Медиана:      1.00 bands
  95 перцентиль: 2.00 bands

GRA:
  Макс. ошибка: 2.0 bands
  Мин. ошибка:  0.0 bands
  Медиана:      1.00 bands
  95 перцентиль: 2.00 bands

Overall:
  Макс. ошибка: 2.0 bands
  Мин. ошибка:  0.0 bands
  Медиана:      1.00 bands
  95 перцентиль: 2.00 bands


In [42]:
#  вытащим примеры работы модели
valid_overall = test_df.dropna(subset=['pred_Overall'])
valid_overall['error'] = abs(valid_overall['Overall_Band'] - valid_overall['pred_Overall'])


print("\n лучшие варианты")
best = valid_overall.nsmallest(3, 'error')

for idx in best.index:
    row = test_df.loc[idx]

    print(f"Эссе: {row['essay']}...")
    print(f"\nИстинные оценки:")
    print(f"  TR: {row['TR_Band']:.1f}, CC: {row['CC_Band']:.1f}, LR: {row['LR_Band']:.1f}, GRA: {row['GRA_Band']:.1f}")
    print(f"  Overall: {row['Overall_Band']:.1f}")
    print(f"\nПредсказанные оценки:")
    print(f"  TR: {row['pred_TR']:.1f}, CC: {row['pred_CC']:.1f}, LR: {row['pred_LR']:.1f}, GRA: {row['pred_GRA']:.1f}")
    print(f"  Overall: {row['pred_Overall']:.1f}")
    print(f"\n Ошибка Overall: {row['error']:.2f} bands")


print("\n\n худшие варианты")
worst = valid_overall.nlargest(3, 'error')

for idx in worst.index:
    row = test_df.loc[idx]
    print(f"Эссе: {row['essay']}...")
    print(f"\nИстинные оценки:")
    print(f"  TR: {row['TR_Band']:.1f}, CC: {row['CC_Band']:.1f}, LR: {row['LR_Band']:.1f}, GRA: {row['GRA_Band']:.1f}")
    print(f"  Overall: {row['Overall_Band']:.1f}")
    print(f"\nПредсказанные оценки:")
    print(f"  TR: {row['pred_TR']:.1f}, CC: {row['pred_CC']:.1f}, LR: {row['pred_LR']:.1f}, GRA: {row['pred_GRA']:.1f}")
    print(f"  Overall: {row['pred_Overall']:.1f}")
    print(f"\n Ошибка Overall: {row['error']:.2f} bands")


test_df.to_csv('ielts_predictions.csv', index=False)
metrics_df.to_csv('ielts_metrics.csv')



 лучшие варианты
Эссе: It is not uncommon that many old people today are choosing to live in retirement communities. In this essay, I intend to discuss the main reasons, and I believe that it is a negative development.

On the other hand, people can benefit significantly if they choose to live in retirement centres. The main reason for this view is that retirement community is a place where residents can be able to access to various infrastructures and facilities such as restaurants, swimming pool and multi-functional rooms. Compare to living at home, they are more likely to have a high-standard living and enjoy a better life. Another reason for this view is that as registered nurses and support workers available in retirement centres are usually high-quality, professional and well-trained, older people as vulnerable group are more likely to receive prompt care and treatment when they get sick, especially during emergent situations. 

On the other hand, I tend to believe the problems 

## Итоги:
Неплохо решаем задачу предсказывания оценки с помощью промптинга, дальше псмотрим получится ли улучшить качество и сможем ли мы как-то оценить еще и фидбек от модели?